In [1]:
from skfp.preprocessing import MolFromSmilesTransformer
from skfp.fingerprints import MordredFingerprint
from rdkit import Chem
from rdkit.Chem.AllChem import EmbedMolecule, ETKDGv3, MMFFGetMoleculeProperties
from rdkit.Chem.rdForceFieldHelpers import MMFFGetMoleculeForceField
import pandas as pd

/home/kimariyb/env/miniconda3/envs/chem/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def optimize_mol(mol, max_iter=2000):
    # 1. 先创建mol副本（避免修改原对象）
    mol_copy = Chem.Mol(mol)

    # 2. 加氢原子（构象优化必须步骤，不管原mol是否加氢）
    mol_copy = Chem.AddHs(mol_copy)

    # 3. 生成初始3D构象（已有mol默认是2D，必须生成3D才能优化）
    try:
        EmbedMolecule(mol_copy, ETKDGv3())  # ETKDG算法成功率高
    except Exception as e:
        return None, f"初始3D构象生成失败: {str(e)[:50]}"

    # 4. 构象优化
    try:
        # MMFF94力场（有机小分子首选）
        ff = MMFFGetMoleculeForceField(
            mol_copy, MMFFGetMoleculeProperties(mol_copy)
        )
        ff.Minimize(maxIts=max_iter)
        return mol_copy, "MMFF94优化成功"
    except Exception as e:
        return None, f"构象优化失败: {str(e)[:50]}"

In [4]:
df = pd.read_csv('./alkynes_smi.csv')
smiles_list = df['SMILES'].tolist()

mols = MolFromSmilesTransformer(
    n_jobs=-1, verbose=1
).fit_transform(smiles_list)

n_mols = len(mols)
print(f"原始mols数量：{n_mols}")

66it [00:04, 14.26it/s]                        

原始mols数量：1312


In [5]:
from tqdm.auto import tqdm

# 批量优化
optimized_mols = []
status_list = []

print("开始优化 mols...")
for mol in tqdm(mols):
    if mol is None:  # 跳过无效mol
        optimized_mols.append(None)
        status_list.append("原mol对象无效")
        continue
    opt_mol, status = optimize_mol(mol)
    optimized_mols.append(opt_mol)
    status_list.append(status)

# 最终结果：optimized_mols 就是你要的优化后mols列表
print("\n优化完成！优化后有效mols数量：", len([m for m in optimized_mols if m is not None]))

开始优化 mols...


100%|██████████| 1312/1312 [00:10<00:00, 124.17it/s]


优化完成！优化后有效mols数量： 1312


In [6]:
fp_mordred = MordredFingerprint(use_3D=True, n_jobs=-1, verbose=1)
X_mordred = fp_mordred.transform(optimized_mols)

print(f"Mordred shape: {X_mordred.shape}")
print(f"Mordred example values: {X_mordred[0, :10]}")

66it [00:04, 14.37it/s]                        

Mordred shape: (1312, 1826)
Mordred example values: [4.5303698 5.004088  0.        0.        7.6629877 2.0528808 4.1057615
 7.6629877 1.0947126 2.7663171]


In [7]:
df_mordred = pd.DataFrame(X_mordred, columns=fp_mordred.get_feature_names_out(), index=smiles_list)

# 找到完全没有空值的列
keep_columns = df_mordred.columns[df_mordred.notnull().all()]

# 只保留这些列
X_fp_cleaned = df_mordred[keep_columns].copy()

# 输出清洗结果
print("===== 空值列清洗结果 =====")
print(f"清洗前列数：{df_mordred.shape[1]}")
print(f"清洗后列数：{X_fp_cleaned.shape[1]}")
print(f"删除的列数：{df_mordred.shape[1] - X_fp_cleaned.shape[1]}")
print(f"剩余空值总数：{X_fp_cleaned.isnull().sum().sum()}")


===== 空值列清洗结果 =====
清洗前列数：1826
清洗后列数：1502
删除的列数：324
剩余空值总数：0


In [9]:
X_fp_cleaned.to_csv("./alkynes_dataset_mordred.csv")